# Notebook04 workflow definitions
Purpose: bounded diagnostics, engineering selection, exact recovery, paired training and frozen evaluation. Inputs: prior audited Food101 run and Notebook04 protocol. Outputs: artifacts/notebook_04 and checkpoints/<run_id>. Definitions only: under one second; the main consumer launches a detached worker with a persistent three-hour maximum. Resume: open Notebook04, keep the same run ID and frozen sources, and Run All.

In [ ]:
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
from pathlib import Path
import nbformat
ROOT=Path("/home/benjamin/workspace/ahr_go/fmm_jeppa")
_boot={}
for _cell in nbformat.read(ROOT/"fmm_jepa/bootstrap.ipynb",as_version=4).cells:
    if _cell.cell_type=="code" and "ahr-export" in _cell.metadata.get("tags",[]):
        exec(compile(_cell.source,"bootstrap#ahr-export","exec"),_boot)
_boot["install_notebook_package"](ROOT)


In [ ]:
"""Bounded Notebook04 orchestration; all experimental code stays in notebooks."""
import copy, hashlib, importlib.util, json, math, os, shlex, subprocess, sys, time
from pathlib import Path
import nbformat
from fmm_jepa.training import atomic_json, definition_hashes, run_writer
from fmm_jepa.semantic_pilot import configure_environment, deadline_guard, reuse_food_bundle, session_exists

MAIN_NOTEBOOK='notebooks/04_Bounded_Semantic_Followup.ipynb'
MODULE_FOLDER='experiments/04_followup'

def load_followup(root,name):
    path=Path(root)/MODULE_FOLDER/(name+'.ipynb')
    boot={}
    for cell in nbformat.read(Path(root)/'fmm_jepa/bootstrap.ipynb',as_version=4).cells:
        if cell.cell_type=='code' and 'ahr-export' in cell.metadata.get('tags',[]):
            exec(compile(cell.source,'bootstrap#ahr-export','exec'),boot)
    module_name='fmm_followup_'+name
    spec=importlib.util.spec_from_loader(module_name,boot['TaggedNotebookLoader'](path))
    module=importlib.util.module_from_spec(spec);sys.modules[module_name]=module;spec.loader.exec_module(module)
    return module

def _json(path):return json.loads(Path(path).read_text())

def _digest(value):return hashlib.sha256(json.dumps(value,sort_keys=True,separators=(',',':')).encode()).hexdigest()

def followup_sources(root):
    root=Path(root);result=definition_hashes(root)
    for path in sorted((root/MODULE_FOLDER).glob('*.ipynb')):
        cells=nbformat.read(path,as_version=4).cells
        sources=[cell.source for cell in cells if cell.cell_type=='code' and 'ahr-export' in cell.metadata.get('tags',[])]
        result[str(path.relative_to(root))]=hashlib.sha256('\n'.join(sources).encode()).hexdigest()
    return result

def consumer_hash(root):
    cells=nbformat.read(Path(root)/MAIN_NOTEBOOK,as_version=4).cells
    return _digest([cell.source for cell in cells if cell.cell_type=='code'])

def set_followup_status(artifact,stage,status,**details):
    value={'stage':stage,'status':status,'updated_unix':time.time(),**details}
    atomic_json(Path(artifact)/'pilot_status.json',value);return value

def base_followup_config(root):
    config=_json(Path(root)/'artifacts/notebook_03/semantic-local-pilot-002/resolved_config.json')
    config=copy.deepcopy(config)
    config['probe']['selection']='fixed_final_epoch'
    config['probe']['epochs']=20
    config['training']['checkpoint_every']=50
    config['evaluation']['heldout_enabled']=False
    return configure_environment(config)

def prepare_followup(root,run_id):
    root=Path(root).resolve();artifact=root/'artifacts/notebook_04'/run_id;artifact.mkdir(parents=True,exist_ok=True)
    requested={'schema':'semantic-followup-request/v1','run_id':run_id,'seeds':[20260926,20260927,20260928],
      'saved_source_run':'semantic-local-pilot-002','probe_epoch':20,'diagnostic_trigger_pp':0.5,
      'diagnostic_rule':'If either online-minus-teacher fixed-epoch20 top1 point >=0.5pp, compare EMA starts .996/.99 in both architectures; otherwise semantic feature32/64 plus ViT.',
      'effective_batch':128,'engineering_microbatches':[16,32,64,128],'max_seconds':10800,
      'resume_notebook':MAIN_NOTEBOOK,'resume_cell':'launch-bounded-followup'}
    request=artifact/'requested_protocol.json'
    if request.exists() and _json(request)!=requested:raise ValueError('Run ID requested protocol changed')
    atomic_json(request,requested)
    clock=artifact/'budget_clock.json'
    if not clock.exists():
        now=time.time();atomic_json(clock,{'started_unix':now,'deadline_unix':now+10800,'target_deadline_unix':now+7200,
            'scope':'Notebook04 preparation, diagnostics, calibration, preflight, training and evaluation; persisted three-hour maximum.'})
    old=artifact/'predecessor_file_hashes.json'
    if not old.exists():atomic_json(old,{str(p.relative_to(root)):hashlib.sha256(p.read_bytes()).hexdigest() for p in (root/'fmm_jepa').glob('*.ipynb')})
    for name,digest in _json(old).items():
        if hashlib.sha256((root/name).read_bytes()).hexdigest()!=digest:raise ValueError('Frozen predecessor changed: '+name)
    return {'artifact':str(artifact),'clock':_json(clock),'request':requested}

def make_followup_plan(root,artifact,diagnostic,engineering):
    root=Path(root);artifact=Path(artifact);clock=_json(artifact/'budget_clock.json');config=base_followup_config(root)
    # Decision extraction is finalized against the diagnostic's documented schema.
    factor='ema' if diagnostic['ema_triggered'] else 'feature_rank'
    micro=int(engineering['selected_microbatch']);acc=int(engineering['selected_accumulation_steps'])
    if micro*acc!=128:raise ValueError('Engineering changed effective batch')
    config['training'].update(microbatch=micro,accumulation_steps=acc)
    config['evaluation']['batch_size']=max(16,min(64,micro))
    conditions=[('vit','ema0996',.996,32),('semantic','ema0996',.996,32),('vit','ema0990',.99,32),('semantic','ema0990',.99,32)] if factor=='ema' else [('vit','reference',.996,32),('semantic','features32',.996,32),('semantic','features64',.996,64)]
    measured=engineering['selected_model_step_seconds']
    step_sum=3*sum(float(measured[kind])*(1.20 if features==64 else 1.0) for kind,condition,ema,features in conditions)
    evaluation_reserve=1800.;preflight_reserve=480.;report_reserve=180.;margin=1.25
    usable_target=max(0.,clock['target_deadline_unix']-time.time()-evaluation_reserve-preflight_reserve-report_reserve)
    usable_hard=max(0.,clock['deadline_unix']-time.time()-evaluation_reserve-preflight_reserve-report_reserve)
    target_updates=int(usable_target/(margin*step_sum)//100)*100
    hard_updates=int(usable_hard/(margin*step_sum)//100)*100
    # Prefer at least the prior 500-update horizon when it fits the hard cap.
    updates=min(1500,hard_updates,max(500,target_updates))
    if updates<100:raise TimeoutError('Insufficient time for a complete paired training screen plus evaluation')
    arms=[]
    for seed_index,seed in enumerate((20260926,20260927,20260928)):
        for kind,condition,ema,features in conditions:
            c=copy.deepcopy(config);c['training'].update(updates=updates,ema_start=ema)
            c['seeds'].update(model_init=seed,data_order=1101+10000*seed_index,augmentation=2202+10000*seed_index,mask=3303+10000*seed_index,probe=4404)
            c['model'].update(semantic_features=features,feature_seed=seed+1000)
            c['evaluation']['diagnostic_steps']=[0,updates]
            arms.append({'id':f's{seed}-{kind}-{condition}','kind':kind,'seed':seed,'condition':condition,'config':c})
    plan={'schema':'semantic-followup-plan/v1','run_id':artifact.name,'factor':factor,'updates':updates,'round_updates':100,
      'seeds':[20260926,20260927,20260928],'arms':arms,'source_hashes':followup_sources(root),'consumer_notebook_sha256':consumer_hash(root),
      'resume_notebook':MAIN_NOTEBOOK,'resume_cell':'launch-bounded-followup','evaluation_reserve_seconds':evaluation_reserve,
      'pessimistic_seconds_per_round_update':margin*step_sum,'budget':{'clock':clock,'target_updates_by_timing':target_updates,
       'hard_updates_by_timing':hard_updates,'preferred_minimum_updates':500,'timing_margin':margin,'predicted_training_seconds':updates*step_sum,
       'evaluation_reserve_seconds':evaluation_reserve,'preflight_reserve_seconds':preflight_reserve,'report_reserve_seconds':report_reserve},
      'diagnostic_selection':diagnostic,'engineering_selection':engineering,
      'evaluation_selection':'fixed final probe epoch20; no best-development checkpoint selection',
      'uncertainty':'Three paired training seeds; image-bootstrap intervals are conditional and do not replace seed uncertainty.',
      'heldout_status':'NOT_RUN'}
    return plan

def check_followup_plan(root,plan):
    if plan['source_hashes']!=followup_sources(root):raise ValueError('Frozen follow-up definitions changed')
    if plan['consumer_notebook_sha256']!=consumer_hash(root):raise ValueError('Frozen main notebook code changed')

def run_bounded_followup(root,run_id):
    root=Path(root).resolve();prepared=prepare_followup(root,run_id);artifact=Path(prepared['artifact']);clock=prepared['clock'];deadline=clock['deadline_unix']
    path=artifact/'pilot_status.json'
    if path.exists() and _json(path).get('status')=='COMPLETED':
        check_followup_plan(root,_json(artifact/'plan.json'));return _json(path)
    if time.time()>=deadline:raise TimeoutError('Authorized follow-up clock expired; evidence retained')
    with run_writer(artifact),deadline_guard(deadline):
        try:
            diagnostic_module=load_followup(root,'diagnostics');engineering_module=load_followup(root,'engineering')
            set_followup_status(artifact,'saved_encoder_diagnostics','RUNNING')
            diagnostic=diagnostic_module.run_saved_encoder_diagnostics(root,artifact,deadline)
            atomic_json(artifact/'diagnostic_result.json',diagnostic)
            set_followup_status(artifact,'engineering_screen','RUNNING')
            engineering=engineering_module.run_engineering_screen(root,artifact,deadline)
            atomic_json(artifact/'engineering_result.json',engineering)
            ready_path=artifact/'implementation_ready.json'
            if not ready_path.exists():set_followup_status(artifact,'source_validation','RUNNING')
            while not ready_path.exists():
                if time.time()>=deadline-1800:raise TimeoutError('Implementation validation not ready within experiment budget')
                time.sleep(1)
            ready=_json(ready_path)
            if ready['source_hashes']!=followup_sources(root) or ready['consumer_notebook_sha256']!=consumer_hash(root):
                raise ValueError('Validated implementation source identity changed')
            plan_path=artifact/'plan.json'
            if plan_path.exists():plan=_json(plan_path);check_followup_plan(root,plan)
            else:plan=make_followup_plan(root,artifact,diagnostic,engineering);atomic_json(plan_path,plan)
            runner=load_followup(root,'runner')
            bundle=reuse_food_bundle(root,plan['arms'][0]['config'],artifact,verify_images=True)
            set_followup_status(artifact,'exact_resume_preflight','RUNNING',factor=plan['factor'],updates=plan['updates'],arms=len(plan['arms']))
            runner.prepare_factorial(root,artifact,plan,bundle)
            certificate=runner.verify_factorial_resume(root,artifact,plan,bundle,deadline)
            atomic_json(artifact/'preflight_result.json',certificate)
            set_followup_status(artifact,'factorial_training','RUNNING',factor=plan['factor'],updates=plan['updates'],arms=len(plan['arms']))
            training=runner.run_factorial_training(root,artifact,plan,bundle,deadline)
            atomic_json(artifact/'training_result.json',training)
            set_followup_status(artifact,'frozen_feature_evaluation','RUNNING',factor=plan['factor'])
            evaluation=runner.evaluate_factorial(root,artifact,plan,bundle,deadline)
            atomic_json(artifact/'evaluation_result.json',evaluation)
            if evaluation.get('status') not in ('COMPLETED','COMPLETE','PASS'):raise RuntimeError('Factorial evaluation incomplete')
            check_followup_plan(root,plan)
            return set_followup_status(artifact,'complete','COMPLETED',factor=plan['factor'],updates=plan['updates'],arms=len(plan['arms']),elapsed_seconds=time.time()-clock['started_unix'])
        except BaseException as error:
            set_followup_status(artifact,'interrupted','INCOMPLETE' if isinstance(error,TimeoutError) else 'FAILED',error=type(error).__name__+': '+str(error),elapsed_seconds=time.time()-clock['started_unix'])
            raise

def launch_bounded_followup(root,run_id):
    root=Path(root).resolve();prepared=prepare_followup(root,run_id);artifact=Path(prepared['artifact']);session='fmm-followup-'+run_id
    if session_exists(session):
        load_followup(root,'deadline_watchdog').launch_deadline_watchdog(root,run_id)
        return {'status':'EXISTING_SESSION','session':session,'attach':'tmux attach -t '+session}
    if (artifact/'pilot_status.json').exists() and _json(artifact/'pilot_status.json').get('status')=='COMPLETED':
        check_followup_plan(root,_json(artifact/'plan.json'));return {'status':'COMPLETED','artifact':str(artifact)}
    if time.time()>=prepared['clock']['deadline_unix']:raise TimeoutError('No time remains; no automatic deadline extension')
    setup=next(c.source for c in nbformat.read(root/MAIN_NOTEBOOK,as_version=4).cells if c.get('id')=='bootstrap-bounded-followup')
    code=setup+'\nWORKFLOW.run_bounded_followup(ROOT, '+repr(run_id)+')\n'
    worker=artifact/'worker.ipynb'
    if worker.exists():
        if [c.source for c in nbformat.read(worker,as_version=4).cells if c.cell_type=='code']!=[code]:raise ValueError('Worker code changed')
    else:
        n=nbformat.v4.new_notebook(cells=[nbformat.v4.new_markdown_cell('# Notebook04 detached worker\nResume through '+MAIN_NOTEBOOK+' with the same run ID. Full-state checkpoints are under checkpoints/'+run_id+'.'),nbformat.v4.new_code_cell(code)],metadata={'kernelspec':{'display_name':'Python (ahr-go)','language':'python','name':'ahr-go'}});nbformat.write(n,worker)
    command=[sys.executable,'-m','jupyter','nbconvert','--execute','--to','notebook','--inplace','--ExecutePreprocessor.timeout=-1','--ExecutePreprocessor.kernel_name=ahr-go',str(worker)]
    shell='exec '+shlex.join(command)+' > '+shlex.quote(str(artifact/'worker.log'))+' 2>&1'
    atomic_json(artifact/'launch.json',{'session':session,'command':command,'attach':'tmux attach -t '+session,'detach':'Ctrl-b, then d','deadline_unix':prepared['clock']['deadline_unix']})
    subprocess.run(['tmux','new-session','-d','-s',session,'-c',str(root),'bash','-c',shell],check=True)
    load_followup(root,'deadline_watchdog').launch_deadline_watchdog(root,run_id)
    return {'status':'LAUNCHED','session':session,'attach':'tmux attach -t '+session}

def monitor_bounded_followup(root,run_id,poll_seconds=10):
    from IPython.display import display,Markdown
    artifact=Path(root)/'artifacts/notebook_04'/run_id;session='fmm-followup-'+run_id;missing=None
    handle=display(Markdown('Connecting to detached follow-up…'),display_id=True)
    while True:
        path=artifact/'pilot_status.json';s=_json(path) if path.exists() else {'status':'STARTING','stage':'startup'}
        handle.update(Markdown('**'+s['status']+'** — '+s.get('stage','')+'; saved evidence: `'+str(artifact)+'`.'))
        if s['status']=='COMPLETED':return s
        if s['status'] in ('FAILED','INCOMPLETE'):raise RuntimeError(s)
        if not session_exists(session):
            missing=missing or time.time()
            if time.time()-missing>30:raise RuntimeError('Worker session exited without completion; inspect worker.log and saved status')
        else:missing=None
        time.sleep(min(60,max(1,poll_seconds)))
